# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load and analyze the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. The dataset contains ordered logistic regression outputs on adoption predictors of indigenous and modern knowledge in rangeland management practices among pastoral households in Northern Kenya.

### Dataset Source
This dataset is described by a [Croissant schema](https://github.com/mlcommons/croissant) accessible via the following URL:

```
https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json
```


In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install -U mlcroissant

## 1. Data Loading
Load the dataset metadata and list available record sets using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Define Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
meta = dataset.metadata

print(f"Dataset title: {meta.name if hasattr(meta, 'name') else ''}")
print(f"Description: {meta.description if hasattr(meta, 'description') else ''}\n")

## 2. Data Overview
Explore the record sets and their schema definitions. Each entity (record set, field, column, etc.) is identified by its `@id`.

Let's enumerate the available record sets and their fields.

In [ ]:
# List all record set @id's with their labels and fields
record_set_ids = [r['@id'] for r in dataset.schema.get('recordSet', [])]

if not record_set_ids:
    print("No record sets found in schema. Attempting to infer record sets from data distribution...")
    # Try fetching from dataset itself in case schema is sparse (common in some exports)
    record_sets = list(dataset.record_sets())
    record_set_ids = [rs['@id'] for rs in record_sets]
    for rs in record_sets:
        print(f"- Record set @id: {rs['@id']}")
        fields = [f['@id'] for f in rs.get('field', [])]
        print(f"  Fields: {fields}")
else:
    for rs in dataset.schema['recordSet']:
        label = rs.get('name', rs.get('@id', None))
        print(f"- Record set @id: {rs['@id']}, label: {label}")
        fields = [f['@id'] for f in rs.get('field', [])]
        print(f"  Fields: {fields}")

## 3. Data Extraction
Load the data from each record set into a pandas DataFrame. Reference each entity by its full `@id`.

**Note:** If you have a specific record set `@id` with known data (from the code above), use it directly. Otherwise, the following will try to load all found record sets.

In [ ]:
# Identify record sets for data loading
all_record_sets = list(dataset.record_sets())
record_set_ids = [rs['@id'] for rs in all_record_sets]

dataframes = dict()
for rec_id in record_set_ids:
    try:
        records = list(dataset.records(record_set=rec_id))
        df = pd.DataFrame(records)
        dataframes[rec_id] = df
        print(f"Loaded record set: {rec_id} | {len(df)} rows, {len(df.columns)} columns.")
    except Exception as e:
        print(f"Failed to load record set {rec_id}: {e}")

# Show columns of the first loaded record set (or placeholder if none)
if dataframes:
    first_id = list(dataframes.keys())[0]
    print(f"\nColumns for record set '@id': {first_id}")
    print(dataframes[first_id].columns.tolist())
    display(dataframes[first_id].head())
else:
    print("No dataframes loaded.")

## 4. Exploratory Data Analysis (EDA)
Perform initial EDA by filtering numeric columns, normalizing data, and grouping by categorical fields.

First, select a numeric field by its `@id`. If no numeric fields are present, you may need to inspect the loaded dataframe to pick a relevant column.

In [ ]:
# Adjust the following @ids to match your data schema as discovered in Data Overview.

# For demonstration, let's try to autodetect numeric fields and group fields in the first record set:
if dataframes:
    df = list(dataframes.values())[0]
    df_id = list(dataframes.keys())[0]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    group_candidate_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    print(f"Numeric candidate fields (@id): {numeric_cols}")
    print(f"Grouping candidate fields (@id): {group_candidate_cols}")

    if numeric_cols:
        numeric_field = numeric_cols[0]
        threshold = df[numeric_field].mean()  # Example threshold: mean
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered records where '{numeric_field}' > {threshold:.2f}")
        print(filtered_df.head())

        # Normalization
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"\nNormalized '{numeric_field}':")
        print(filtered_df[[numeric_field, norm_col]].head())

        # Grouping on a categorical field if available
        if group_candidate_cols:
            group_field = group_candidate_cols[0]
            grouped = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"\nGrouped '{numeric_field}' mean by '{group_field}':")
            print(grouped.head())
    else:
        print("No numeric fields found for analysis.")
else:
    print("No data available to perform EDA.")

## 5. Visualization
Create basic visualizations of the data, such as histograms, boxplots, or bar charts, using the selected numeric and group fields. Modify the field names/@ids as needed.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if dataframes:
    df = list(dataframes.values())[0]
    numeric_cols = df.select_dtypes(include=['number']).columns.tolist()
    group_cols = df.select_dtypes(include=['object', 'category']).columns.tolist()
    if numeric_cols:
        num_field = numeric_cols[0]
        # Histogram
        plt.figure(figsize=(8,4))
        sns.histplot(df[num_field], bins=30, kde=True, color='c')
        plt.title(f"Distribution of {num_field} (by @id)")
        plt.xlabel(num_field)
        plt.show()
        # Boxplot grouped by a categorical variable (if any)
        if group_cols:
            group_field = group_cols[0]
            plt.figure(figsize=(10,4))
            sns.boxplot(x=group_field, y=num_field, data=df)
            plt.title(f"{num_field} by {group_field}")
            plt.xticks(rotation=30)
            plt.show()
    else:
        print("No numeric fields available for visualization.")
else:
    print("Data not available for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, inspect, and analyze the FAIR² dataset using `mlcroissant`. We identified available record sets, loaded records by referencing their `@id`, and performed basic exploratory data analysis and visualization on available numeric and categorical fields.

**Key Steps:**
- Data is referenced and manipulated by entity `@id` for consistency and schema transparency.
- Explored the schema, loaded relevant tables, filtered and normalized sample fields.
- Provided a foundation to conduct further domain analysis on the drivers of knowledge adoption in Northern Kenya rangeland management practices.
